# Chapter 7: Fine-Tuning to Follow Instructions 🎯

The final chapter — making the model actually useful through instruction tuning.

**What we'll do:**
1. Create an instruction dataset (Alpaca format)
2. Instruction-tune a small GPT model
3. Compare before vs after
4. Interactive chat

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import tiktoken
import matplotlib.pyplot as plt

from ch04.gpt_model import GPTModel
from ch04.config import GPTConfig
from ch07.instruction_dataset import (
    INSTRUCTIONS, format_instruction, create_dataloaders, InstructionDataset
)
from ch07.instruction_tuning import (
    train_instruction_model, generate_response, calc_loss_loader
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Explore the Instruction Dataset

We use a synthetic dataset of ~60 instruction-response pairs in Alpaca format.

In [ ]:
print(f'Total instructions: {len(INSTRUCTIONS)}\n')

# Show category distribution
categories = {
    'Q&A': [i for i in INSTRUCTIONS if '?' in i['instruction'] and 'translate' not in i['instruction'].lower()],
    'Translation': [i for i in INSTRUCTIONS if 'translat' in i['instruction'].lower() or 'how do you say' in i['instruction'].lower()],
    'Summarization': [i for i in INSTRUCTIONS if 'summar' in i['instruction'].lower()],
    'Creative': [i for i in INSTRUCTIONS if any(w in i['instruction'].lower() for w in ['write', 'haiku', 'poem', 'story', 'limerick'])],
}
for cat, items in categories.items():
    print(f'  {cat}: {len(items)} examples')

In [ ]:
# Show formatted examples
for i in [0, 12, 22, 30]:
    print(f'--- Example {i+1} ---')
    print(format_instruction(INSTRUCTIONS[i]))
    print()

In [ ]:
# Create DataLoaders
train_loader, val_loader, tokenizer = create_dataloaders(
    max_length=256,
    batch_size=4,
)

# Peek at token lengths
lengths = []
for entry in INSTRUCTIONS:
    text = format_instruction(entry)
    tokens = tokenizer.encode(text)
    lengths.append(len(tokens))

plt.figure(figsize=(8, 3))
plt.hist(lengths, bins=20, edgecolor='black', alpha=0.7)
plt.xlabel('Token Length')
plt.ylabel('Count')
plt.title('Distribution of Instruction Token Lengths')
plt.axvline(x=256, color='r', linestyle='--', label='max_length=256')
plt.legend()
plt.tight_layout()
plt.show()
print(f'Min: {min(lengths)}, Max: {max(lengths)}, Mean: {sum(lengths)/len(lengths):.0f}')

## 2. Create the Model

Small config for CPU-friendly training.

In [ ]:
cfg = GPTConfig(
    vocab_size=50_257,
    d_model=128,       # Small for demo
    n_heads=4,
    n_layers=4,
    max_seq_len=256,
    dropout=0.1,
)

model = GPTModel(cfg)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')
print(f'Config: d_model={cfg.d_model}, layers={cfg.n_layers}, heads={cfg.n_heads}')

## 3. Before Training: Random Model Output

Let's see what the model generates **before** instruction tuning — it should be random gibberish.

In [ ]:
test_instructions = [
    'What is the capital of France?',
    'Write a haiku about rain.',
    'Explain what a neural network is in simple terms.',
    'Translate hello to Spanish.',
    'What is 15 multiplied by 7?',
]

print('=' * 60)
print('BEFORE Instruction Tuning (random weights)')
print('=' * 60)

# IMPORTANT: model.to(device) before generation!
model.to(device)

for instr in test_instructions:
    response = generate_response(
        model, instr, tokenizer,
        device=device, max_new_tokens=50,
    )
    print(f'\n📝 {instr}')
    print(f'🤖 {response[:200]}')

## 4. Instruction Tuning

Train the model on our instruction dataset. This teaches the model to:
1. Recognize the instruction format
2. Generate relevant responses

With ~60 examples and a tiny model, the model will learn the **format** and **some patterns**, but won't have deep knowledge.

In [ ]:
# Measure initial loss
model.to(device)
initial_train_loss = calc_loss_loader(model, train_loader, device)
initial_val_loss = calc_loss_loader(model, val_loader, device)
print(f'Initial train loss: {initial_train_loss:.4f}')
print(f'Initial val loss:   {initial_val_loss:.4f}')

In [ ]:
# Train!
history = train_instruction_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer=tokenizer,
    num_epochs=15,
    max_lr=5e-4,
    eval_every=20,
    device=device,
)

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Training loss (per step)
ax1.plot(history['train_losses'], alpha=0.3, label='Train (per step)')
# Smoothed
window = 10
if len(history['train_losses']) > window:
    smoothed = [sum(history['train_losses'][max(0,i-window):i+1]) / min(i+1, window)
                for i in range(len(history['train_losses']))]
    ax1.plot(smoothed, label=f'Train (smoothed, w={window})', color='blue')
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Learning rate
ax2.plot(history['lrs'])
ax2.set_xlabel('Step')
ax2.set_ylabel('Learning Rate')
ax2.set_title('Learning Rate Schedule')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. After Training: Compare Results

Now let's see how the model responds to the same instructions after training.

In [ ]:
print('=' * 60)
print('AFTER Instruction Tuning')
print('=' * 60)

model.to(device)

for instr in test_instructions:
    response = generate_response(
        model, instr, tokenizer,
        device=device, max_new_tokens=80,
    )
    print(f'\n📝 {instr}')
    print(f'🤖 {response[:300]}')

In [ ]:
# Test on instructions NOT in the training set
print('=' * 60)
print('Generalization Test (unseen instructions)')
print('=' * 60)

unseen = [
    'What is the capital of Germany?',
    'Write a haiku about the moon.',
    'What is 12 times 8?',
    'Translate good night to French.',
    'Summarize: Dogs are loyal animals that have been domesticated for thousands of years.',
]

for instr in unseen:
    response = generate_response(
        model, instr, tokenizer,
        device=device, max_new_tokens=80,
    )
    print(f'\n📝 {instr}')
    print(f'🤖 {response[:300]}')

## 6. Interactive Chat

Try chatting with the model! (In a notebook, this runs inline. For a better experience, use `python -m ch07.chat`)

In [ ]:
# Interactive — type your own instructions!
# (Set to a few rounds for notebook; use chat.py for unlimited)

print('Type an instruction (or "quit" to stop):\n')

for _ in range(5):  # 5 rounds in notebook
    try:
        user_input = input('📝 You: ')
    except EOFError:
        break
    
    if user_input.lower() in ('quit', 'exit', 'q', ''):
        break
    
    response = generate_response(
        model, user_input, tokenizer,
        device=device, max_new_tokens=100,
    )
    print(f'🤖 Bot: {response}\n')

print('\n👋 Chat ended.')

## 7. Key Observations

**What the small model learns:**
- The instruction-response **format** (it generates text after `### Response:`)
- Some **patterns** from the training data (e.g., "The capital of X is Y")
- Basic **structure** of answers

**What it doesn't learn (due to small size):**
- Factual knowledge beyond the training examples
- Complex reasoning
- Generalization to very different instruction types

**In a real system:**
- Start with a **pretrained** model (not random weights)
- Use **thousands** of high-quality instruction pairs
- Follow up with **RLHF or DPO** for alignment
- Use **larger models** (7B+ parameters) for real capability

## 🎉 Course Complete!

We've built the full LLM pipeline from scratch:

```
ch02: Text → Tokens (tokenization)
ch03: Attention mechanisms
ch04: GPT architecture
ch05: Pretraining (next-token prediction)
ch06: Fine-tuning for classification
ch07: Fine-tuning for instruction following ← YOU ARE HERE
```